# Home Credit — E02 credit/amount factorial ablation

This notebook decomposes the measured E02-A credit/amount family into three independent factors:

- N: explicitly recompute four overlapping E01 ratios with float32 safe division.
- R: add CREDIT_ANNUITY_RATIO.
- D: add CREDIT_GOODS_DIFF.

It runs all eight N/R/D combinations sequentially on one immutable fold list. E01 target, population, LightGBM parameters, early stopping and ROC-AUC contract remain locked. No auxiliary table, tuning or leaderboard-based selection is used. Default mode is smoke; only baseline mode can select E02-FINAL and write a submission file.


## Reproducibility references

Locked E01 OOF AUC 0.768696 and the completed E02-A OOF AUC 0.769269 are reference values only. They are not inserted into experiment outputs. Every score, delta and chart below is generated from the current run.


In [ ]:
# 1. Clone public repository and import reusable project code
import gc
import importlib
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/ManhTanTran/Qaci-datascience.git"
REPO_BRANCH = "codex/home-credit-e02-ablation"
REPO_COMMIT = None  # Pin a tested SHA for a final archived run.
REPO_DIR = Path("/kaggle/working/Qaci-datascience")

def run_git(*arguments: str) -> None:
    subprocess.run(["git", "-C", str(REPO_DIR), *arguments], check=True)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
run_git("fetch", "--depth", "1", "origin", REPO_BRANCH)
if REPO_COMMIT is None:
    run_git("checkout", REPO_BRANCH)
    run_git("pull", "--ff-only", "origin", REPO_BRANCH)
else:
    run_git("fetch", "--depth", "1", "origin", REPO_COMMIT)
    run_git("checkout", "--detach", REPO_COMMIT)

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
SRC_DIR = (REPO_DIR / "src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()

from credit_scoring.artifacts import export_dataframe_artifact, export_json_artifact
from credit_scoring.data.home_credit import audit_home_credit_data, load_home_credit_data
from credit_scoring.evaluation.cross_validation import create_stratified_folds
from credit_scoring.experiments.home_credit_credit_amount_factorial import (
    compare_e01_to_current_e02_a,
    nrd_reproduces_current_e02_a,
    prepare_credit_amount_factorial_data,
    resolve_credit_amount_factorial_experiments,
    select_e02_final,
)
from credit_scoring.features.home_credit_credit_amount_factorial import (
    describe_credit_amount_factors,
)
from credit_scoring.modeling.lightgbm_model import run_lightgbm_cv
from credit_scoring.reproducibility import set_global_seed
from credit_scoring.submission.home_credit import create_home_credit_submission

set_global_seed(42)
print("Git commit:", GIT_COMMIT)
print("Python:", platform.python_version())


In [ ]:
# 2. Locked configuration — smoke is the default
RUN_MODES = {
    "smoke": {
        "sample_size": 5_000,
        "n_splits": 3,
        "n_estimators": 300,
        "early_stopping_rounds": 50,
    },
    "baseline": {
        "sample_size": None,
        "n_splits": 5,
        "n_estimators": 5_000,
        "early_stopping_rounds": 200,
    },
}
REFERENCE_VALUES = {
    "E01_LOCKED_OOF_AUC": 0.768696,
    "E02_A_OOF_AUC": 0.769269,
}
CONFIG = {
    "experiment_name": "E02_credit_amount_factorial_ablation",
    "run_mode": "smoke",
    "data_dir": "/kaggle/input/competitions/home-credit-default-risk",
    "output_dir": "/kaggle/working/home_credit_outputs",
    "selected_experiments": None,
    "random_state": 42,
    "tie_tolerance": 1e-5,
    "max_std_increase": 5e-4,
}
if CONFIG["run_mode"] not in RUN_MODES:
    raise ValueError(f"Unknown run mode: {CONFIG['run_mode']}")
MODE = RUN_MODES[CONFIG["run_mode"]]
MODEL_CONFIG = {
    "learning_rate": 0.02,
    "n_estimators": MODE["n_estimators"],
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 80,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "random_state": CONFIG["random_state"],
    "n_jobs": -1,
    "verbosity": -1,
}
VALIDATION_CONFIG = {
    "n_splits": MODE["n_splits"],
    "shuffle": True,
    "random_state": CONFIG["random_state"],
    "early_stopping_rounds": MODE["early_stopping_rounds"],
    "keep_models": False,
}
EXPERIMENTS = resolve_credit_amount_factorial_experiments(
    CONFIG["selected_experiments"]
)
OUTPUT_DIR = (
    Path(CONFIG["output_dir"])
    / CONFIG["experiment_name"]
    / CONFIG["run_mode"]
    / GIT_COMMIT[:8]
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame([
    {"experiment": name, "factors": "".join(factors) or "none"}
    for name, factors in EXPERIMENTS.items()
]))


In [ ]:
# 3. Load only application_train and application_test
data = load_home_credit_data(
    CONFIG["data_dir"],
    tables=("application_train", "application_test"),
    nrows=MODE["sample_size"],
    reduce_memory=True,
)
train = data["application_train"]
test = data["application_test"]
display(audit_home_credit_data(data))
assert train["TARGET"].isin([0, 1]).all()
assert train["SK_ID_CURR"].is_unique
assert test["SK_ID_CURR"].is_unique


In [ ]:
# 4. One immutable fold list for every experiment
target = train["TARGET"].astype("int8")
folds = create_stratified_folds(
    target,
    n_splits=VALIDATION_CONFIG["n_splits"],
    shuffle=VALIDATION_CONFIG["shuffle"],
    random_state=VALIDATION_CONFIG["random_state"],
)
fold_assignment = np.full(len(train), -1, dtype=np.int8)
for fold_number, (_, valid_index) in enumerate(folds, start=1):
    fold_assignment[np.asarray(valid_index, dtype=int)] = fold_number
assert np.all(fold_assignment > 0)
assert len(np.unique(fold_assignment)) == VALIDATION_CONFIG["n_splits"]
display(pd.Series(fold_assignment).value_counts().sort_index().rename("validation_rows"))


In [ ]:
# 5. Diagnose E01 versus current E02-A and require exact NRD reproduction
train_predictors = train.drop(columns="TARGET")
feature_matrix_differences = compare_e01_to_current_e02_a(train_predictors)
train_nrd_reproduced = nrd_reproduces_current_e02_a(train_predictors)
test_nrd_reproduced = nrd_reproduces_current_e02_a(test)
nrd_reproduced = bool(train_nrd_reproduced and test_nrd_reproduced)
display(feature_matrix_differences)
print("E02-NRD reproduces current E02-A:", nrd_reproduced)
if not nrd_reproduced:
    raise RuntimeError(
        "E02-NRD does not reproduce current E02-A. Stop final selection and inspect builders."
    )
del train_predictors
gc.collect()


In [ ]:
# 6. Run all eight combinations sequentially and export per-experiment artifacts
summary_rows = []
fold_rows = []
artifact_index = {}
baseline_fold_scores = None
baseline_oof_auc = None
baseline_std_auc = None
fold_fingerprint = None

for experiment_name, factors in EXPERIMENTS.items():
    print(f"\nRunning {experiment_name}: {factors or ('E01 locked',)}")
    prepared = prepare_credit_amount_factorial_data(train, test, factors=factors)
    spec = describe_credit_amount_factors(factors)
    result = run_lightgbm_cv(
        prepared.train_features,
        prepared.target,
        prepared.test_features,
        categorical_features=prepared.categorical_features,
        model_config=MODEL_CONFIG,
        validation_config=VALIDATION_CONFIG,
        folds=folds,
    )
    current_fingerprint = str(result["metadata"]["fold_fingerprint"])
    if fold_fingerprint is None:
        fold_fingerprint = current_fingerprint
    assert current_fingerprint == fold_fingerprint
    assert np.all(result["validation_counts"] == 1)
    assert np.isfinite(result["oof_predictions"]).all()
    assert np.isfinite(result["test_predictions"]).all()

    fold_scores = np.asarray(result["fold_scores"], dtype=float)
    if experiment_name == "E01_LOCKED":
        baseline_fold_scores = fold_scores.copy()
        baseline_oof_auc = float(result["oof_auc"])
        baseline_std_auc = float(result["std_auc"])
    fold_deltas = fold_scores - baseline_fold_scores
    summary_row = {
        "experiment": experiment_name,
        "enabled_factors": "".join(spec.factors) or "none",
        "n_features": prepared.train_features.shape[1],
        "n_added_features": len(spec.added_columns),
        "added_features": "|".join(spec.added_columns),
        "n_overwritten_features": len(spec.overwritten_columns),
        "overwritten_features": "|".join(spec.overwritten_columns),
        "mean_fold_auc": float(result["mean_auc"]),
        "std_fold_auc": float(result["std_auc"]),
        "oof_auc": float(result["oof_auc"]),
        "delta_oof_auc_vs_e01": float(result["oof_auc"] - baseline_oof_auc),
        "positive_fold_count_vs_e01": int((fold_deltas > 0).sum()),
        "best_iterations": "|".join(str(int(value)) for value in result["best_iterations"]),
        "runtime_seconds": float(result["runtime"]),
        "fold_fingerprint": current_fingerprint,
    }
    for fold_number, (score, delta, best_iteration) in enumerate(
        zip(fold_scores, fold_deltas, result["best_iterations"], strict=True),
        start=1,
    ):
        summary_row[f"fold_{fold_number}_auc"] = float(score)
        summary_row[f"fold_{fold_number}_delta_vs_e01"] = float(delta)
        fold_rows.append({
            "experiment": experiment_name,
            "enabled_factors": "".join(spec.factors) or "none",
            "fold": fold_number,
            "auc": float(score),
            "delta_auc_vs_e01": float(delta),
            "best_iteration": int(best_iteration),
            "fold_fingerprint": current_fingerprint,
        })
    summary_rows.append(summary_row)

    experiment_dir = OUTPUT_DIR / "experiments" / experiment_name
    feature_manifest = pd.DataFrame({
        "feature": prepared.train_features.columns,
        "dtype": [str(dtype) for dtype in prepared.train_features.dtypes],
        "is_added": [column in spec.added_columns for column in prepared.train_features],
        "is_overwritten": [
            column in spec.overwritten_columns for column in prepared.train_features
        ],
    })
    oof_frame = pd.DataFrame({
        "SK_ID_CURR": prepared.train_ids.to_numpy(),
        "TARGET": prepared.target.to_numpy(),
        "FOLD": fold_assignment,
        "OOF_PREDICTION": result["oof_predictions"],
        "VALIDATION_COUNT": result["validation_counts"],
    })
    test_frame = pd.DataFrame({
        "SK_ID_CURR": prepared.test_ids.to_numpy(),
        "TEST_PREDICTION": result["test_predictions"],
    })
    paths = {
        "feature_manifest": export_dataframe_artifact(
            feature_manifest, experiment_dir / "feature_manifest.csv"
        ),
        "feature_importance": export_dataframe_artifact(
            result["feature_importance"], experiment_dir / "feature_importance.csv"
        ),
        "oof_predictions": export_dataframe_artifact(
            oof_frame, experiment_dir / "oof_predictions.csv"
        ),
        "test_predictions": export_dataframe_artifact(
            test_frame, experiment_dir / "test_predictions.csv"
        ),
    }
    experiment_metadata = {
        **summary_row,
        "feature_dtypes": {
            column: str(dtype)
            for column, dtype in prepared.train_features.dtypes.items()
        },
        "artifact_paths": {name: str(path) for name, path in paths.items()},
    }
    paths["metadata"] = export_json_artifact(
        experiment_metadata, experiment_dir / "experiment_metadata.json"
    )
    artifact_index[experiment_name] = {
        name: str(path) for name, path in paths.items()
    }
    print(
        f"{experiment_name}: OOF={result['oof_auc']:.6f}, "
        f"delta={summary_row['delta_oof_auc_vs_e01']:+.6f}, "
        f"positive folds={summary_row['positive_fold_count_vs_e01']}/{len(folds)}"
    )
    result["fitted_models"].clear()
    del prepared, result, feature_manifest, oof_frame, test_frame
    gc.collect()

factorial_summary = pd.DataFrame(summary_rows)
fold_metrics = pd.DataFrame(fold_rows)
display(factorial_summary.sort_values("oof_auc", ascending=False))


In [ ]:
# 7. Select E02-FINAL only from a successful full baseline run
all_completed = len(factorial_summary) == len(EXPERIMENTS)
selection = None
submission_path = None
if CONFIG["run_mode"] == "baseline" and all_completed and nrd_reproduced:
    selection = select_e02_final(
        factorial_summary,
        n_splits=VALIDATION_CONFIG["n_splits"],
        tie_tolerance=CONFIG["tie_tolerance"],
        max_std_increase=CONFIG["max_std_increase"],
    )
    if selection is not None:
        selected_predictions = pd.read_csv(
            artifact_index[selection.source_experiment]["test_predictions"]
        )
        submission_path = create_home_credit_submission(
            selected_predictions["SK_ID_CURR"],
            selected_predictions["TEST_PREDICTION"],
            OUTPUT_DIR / "E02-FINAL" / "submission.csv",
        )
        export_json_artifact({
            "name": selection.name,
            "source_experiment": selection.source_experiment,
            "factors": selection.factors,
            "oof_auc": selection.oof_auc,
            "positive_fold_count": selection.positive_fold_count,
            "std_fold_auc": selection.std_fold_auc,
            "submission_path": str(submission_path),
        }, OUTPUT_DIR / "E02-FINAL" / "config.json")
print("E02-FINAL selection:", selection)
print("Submission created locally (not submitted):", submission_path)


In [ ]:
# 8. Export common artifacts, environment and actual-results chart
def installed_version(package_name: str) -> str | None:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "git_commit": GIT_COMMIT,
    "packages": {
        name: installed_version(name)
        for name in ["numpy", "pandas", "scikit-learn", "lightgbm", "matplotlib"]
    },
}
summary_path = export_dataframe_artifact(
    factorial_summary, OUTPUT_DIR / "factorial_ablation_summary.csv"
)
fold_path = export_dataframe_artifact(fold_metrics, OUTPUT_DIR / "fold_metrics.csv")
difference_path = export_dataframe_artifact(
    feature_matrix_differences, OUTPUT_DIR / "feature_matrix_differences.csv"
)
config_path = export_json_artifact({
    "config": CONFIG,
    "mode": MODE,
    "model_config": MODEL_CONFIG,
    "validation_config": VALIDATION_CONFIG,
    "experiments": EXPERIMENTS,
    "reference_values_for_reproducibility_only": REFERENCE_VALUES,
}, OUTPUT_DIR / "config.json")
environment_path = export_json_artifact(environment, OUTPUT_DIR / "environment.json")

plot_data = factorial_summary.sort_values("delta_oof_auc_vs_e01")
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2e7d32" if value > 0 else "#c62828" for value in plot_data["delta_oof_auc_vs_e01"]]
ax.barh(plot_data["experiment"], plot_data["delta_oof_auc_vs_e01"], color=colors)
ax.axvline(0.0, color="black", linewidth=1)
ax.set_xlabel("Measured delta global OOF AUC versus current-run E01")
ax.set_title(f"Credit/amount factorial ablation — {CONFIG['run_mode']}")
fig.tight_layout()
chart_path = (OUTPUT_DIR / "delta_oof_auc_vs_e01.png").resolve()
fig.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()

run_metadata = {
    "status": "completed" if all_completed else "incomplete",
    "run_mode": CONFIG["run_mode"],
    "git_commit": GIT_COMMIT,
    "dataset_path": str(Path(CONFIG["data_dir"]).resolve()),
    "n_train": len(train),
    "n_test": len(test),
    "fold_fingerprint": fold_fingerprint,
    "oof_coverage_exactly_once": True,
    "nrd_reproduced_current_e02_a": nrd_reproduced,
    "reference_check_deltas": {
        "E01_LOCKED": float(
            factorial_summary.loc[
                factorial_summary["experiment"].eq("E01_LOCKED"), "oof_auc"
            ].iloc[0] - REFERENCE_VALUES["E01_LOCKED_OOF_AUC"]
        ),
        "E02-NRD": float(
            factorial_summary.loc[
                factorial_summary["experiment"].eq("E02-NRD"), "oof_auc"
            ].iloc[0] - REFERENCE_VALUES["E02_A_OOF_AUC"]
        ),
    },
    "selection": None if selection is None else {
        "name": selection.name,
        "source_experiment": selection.source_experiment,
        "factors": selection.factors,
        "oof_auc": selection.oof_auc,
        "positive_fold_count": selection.positive_fold_count,
        "std_fold_auc": selection.std_fold_auc,
    },
    "submission_path": None if submission_path is None else str(submission_path),
    "artifact_paths": {
        "factorial_summary": str(summary_path),
        "fold_metrics": str(fold_path),
        "feature_matrix_differences": str(difference_path),
        "config": str(config_path),
        "environment": str(environment_path),
        "chart": str(chart_path),
        "experiments": artifact_index,
    },
}
metadata_path = export_json_artifact(run_metadata, OUTPUT_DIR / "run_metadata.json")
print("Output directory:", OUTPUT_DIR)
print("Run metadata:", metadata_path)


## Interpretation rules

Rank by measured global OOF AUC, require improvement in at least four of five baseline folds, inspect paired fold deltas and reject material fold-variance deterioration. If candidates are effectively tied, prefer the smaller feature set and avoid unnecessary E01 overwrite. The notebook does not claim statistical significance.

Smoke outputs only validate execution. Do not write smoke metrics to the experiment log. After a successful baseline run, record E02-FINAL and preserve its artifact directory. The recommended next experiment is E03 = E02-FINAL plus aggregated bureau and bureau_balance features.
